# PDF Parser Nexis Texts 

#### This script is useful for Nexis downloads 

In [2]:
import os
import re
import fitz  # PyMuPDF
import pytesseract
import pandas as pd
from datetime import datetime
from PIL import Image
import PyPDF2


folder_path = r"path_to_your_pdf_folder"
output_csv = "extracted_articles.csv"
outlet = "Example News Outlet"


In [15]:
# Check if folder is accessible
print("Exists:", os.path.exists(folder_path))
print("Is Directory:", os.path.isdir(folder_path))

# List first few files
try:
    files = os.listdir(folder_path)
    print(f"Total files detected: {len(files)}")
    print("First 10 files:", files[:10])
except Exception as e:
    print(f"Error listing files: {e}")

pdf_files = [f for f in os.listdir(folder_path) if f.endswith(('.pdf', '.PDF'))]

print(f"Total PDFs found after fix: {len(pdf_files)}")
print("First 10 PDFs:", pdf_files[:10])


Exists: True
Is Directory: True
Total files detected: 860
First 10 files: ['Irrationaliteit is het punt De handelsoorlog van Donald Trump.PDF', 'Groen, geel, blauw en slordig Kunst Nieuw Parijs.PDF', 'Wel je diploma, niet naar Rome_ Corona De verloren Grand Tour.PDF', 'Het geboortehuis van Herman Gorter essay.PDF', 'Verstrengelde dimensies Holland Festival_ Muziek - Stefan Prins, tussen werkelijkheid en schijn.PDF', 'Propaganda op maat Onderzoek Technologie Politieke campagnes in de 21ste eeuw.PDF', 'â  Ik weet dat jij een bloedhond bentâ Holland Festival - Kings of War.PDF', 'Het ontwijken van coherentie Darkness, as a First Act of Creation(2).PDF', 'Taal is ons ding De kruistocht van Emily Bender.PDF', 'Synchroonzwemmen.PDF']
Total PDFs found after fix: 860
First 10 PDFs: ['Irrationaliteit is het punt De handelsoorlog van Donald Trump.PDF', 'Groen, geel, blauw en slordig Kunst Nieuw Parijs.PDF', 'Wel je diploma, niet naar Rome_ Corona De verloren Grand Tour.PDF', 'Het geboortehuis 

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file using PyPDF2."""
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
    return text


In [17]:
test_pdf = os.path.join(folder_path, pdf_files[0])  # Pick first PDF
print(f"Testing file: {test_pdf}")

try:
    text = extract_text_from_pdf(test_pdf)  # Run extraction
    print("Extracted text (first 500 characters):")
    print(text[:500])  # Show first 500 characters of extracted text
except Exception as e:
    print(f"Error extracting text from {test_pdf}: {e}")


Testing file: /Users/Nguye057/Desktop/Research /HUMAN/GA_pdfs/Irrationaliteit is het punt De handelsoorlog van Donald Trump.PDF
Extracted text (first 500 characters):
Page 1 of 4
Irrationaliteit is het punt De handelsoorlog van Donald Trump
Irrationaliteit is het punt; De handelsoorlog van Donald Trump
De Groene Amsterdammer
17 april 2025
© Copyright 2025 De Groene Amsterdammer All Rights Reserved
Section: IN; Blz. 12
Length: 1873 words
Byline: Casper Thomas
Body
Trumps gejojo met handelstarieven heeft niets te maken met beleid. Het is 'het spektakel van de heroïsche leider 
die zijn capaciteit voor shock and awe tentoonspreidt'.
Er bestaat een wereld waarin 


In [18]:
dutch_to_english_months = {
    "januari": "January", "februari": "February", "maart": "March", "april": "April",
    "mei": "May", "juni": "June", "juli": "July", "augustus": "August",
    "september": "September", "oktober": "October", "november": "November", "december": "December"
}

In [ ]:
def replace_dutch_months(date_string):
    """Replace Dutch month names with English equivalents."""
    for dutch, english in dutch_to_english_months.items():
        if dutch in date_string.lower():
            return date_string.lower().replace(dutch, english)
    return date_string

def extract_text_from_pdf(pdf_path):
    """Extract text from all pages of the PDF."""
    text = ""
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            try:
                text += doc[page_num].get_text()
            except Exception as e:
                print(f"Error extracting text from page {page_num} in {pdf_path}: {e}")
                page = doc.load_page(page_num)
                pix = page.get_pixmap()
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                text += pytesseract.image_to_string(img)
    return text

def clean_text(text):
    """Clean unwanted characters from text."""
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def find_date_in_text(text):
    """Find and extract date from the entire text, ignoring time information."""
    date_patterns = [
        re.compile(r'(\d{1,2})\s+(januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+(\d{4})', re.IGNORECASE),
        re.compile(r'(\d{1,2})\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})', re.IGNORECASE),
        re.compile(r'(\d{1,2})[-/](\d{1,2})[-/](\d{4})'),
        re.compile(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}', re.IGNORECASE),
    ]
    
    text = clean_text(text)
    for pattern in date_patterns:
        match = pattern.search(text)
        if match:
            return match.group().strip()
    return None

def find_authors(lines):
    """
    Extract authors from the text, assuming they appear in lines starting with 'Byline:'.
    """
    for line in lines:
        if line.startswith("Byline:"):
            # Remove "Byline:" and return the remaining string
            return line.replace("Byline:", "").strip()
    return "Unknown Authors"  # Default if no byline is found

def parse_date(date_string):
    """Parse a date string into a datetime object."""
    if date_string:
        date_string = replace_dutch_months(date_string)
        date_formats = ["%d %B %Y", "%d-%m-%Y", "%d/%m/%Y", "%B %d, %Y"]
        for date_format in date_formats:
            try:
                return datetime.strptime(date_string, date_format)
            except ValueError:
                continue
    return None

def find_title_and_outlet(lines):
    """
    Extract the actual headline from the text, then clean it by removing 'Het Financieele Dagblad'.
    """
    print("DEBUG: First few lines of the text:")
    print(lines[:10])

    headline = lines[1].strip() if len(lines) > 1 else "Unknown Title"
    if len(lines) > 2 and lines[2].strip():
        headline += f" {lines[2].strip()}"

    headline = headline.replace(outlet, "").strip()
    return headline, outlet

def parse_pdf_text(text):
    """Parse the text to extract title, outlet, date, authors, and body."""
    lines = text.split('\n')
    title, outlet = find_title_and_outlet(lines)
    date_string = find_date_in_text(text)
    date = parse_date(date_string) if date_string else None
    authors = find_authors(lines)

    body_start_idx = text.find("Body") + len("Body")
    body_end_idx = text.find("Graphic", body_start_idx)
    body_end_idx = text.find("Classification", body_start_idx) if body_end_idx == -1 else body_end_idx
    body_end_idx = len(text) if body_end_idx == -1 else body_end_idx
    body = text[body_start_idx:body_end_idx].strip()

    return title, outlet, date, authors, body

def parse_pdfs_in_folder(folder_path):
    """Parse all PDFs in a folder, skipping unwanted files, and create a DataFrame."""
    data = []
    missing_dates = []
    skip_files = {"Bestanden (500)_doclist.PDF"}  # Set of filenames to skip

    for filename in os.listdir(folder_path):
        if filename.endswith(('.pdf', '.PDF')) and filename not in skip_files:
            path = os.path.join(folder_path, filename)
            text = extract_text_from_pdf(path)
            title, outlet, date, authors, body = parse_pdf_text(text)
            data.append({'title': title, 'outlet': outlet, 'date': date, 'authors': authors, 'body': body})

            if not date:
                missing_dates.append(filename)

    print(f"PDFs missing dates: {missing_dates}")
    return pd.DataFrame(data)


df = parse_pdfs_in_folder(folder_path)

print(df.head())
print(df.shape)

nats_or_nans = df.isna().any(axis=1)
rows_with_nats_or_nans = df[nats_or_nans]

print("Rows with missing values:")
print(rows_with_nats_or_nans.shape)

df_cleaned = df.dropna()
print("Cleaned DataFrame shape:")
print(df_cleaned.shape)

# df_cleaned.to_csv(output_csv, index=False)

df_cleaned


DEBUG: First few lines of the text:
['Page 1 of 4', 'Irrationaliteit is het punt De handelsoorlog van Donald Trump', 'Irrationaliteit is het punt; De handelsoorlog van Donald Trump', 'De Groene Amsterdammer', '17 april 2025', '© Copyright 2025 De Groene Amsterdammer All Rights Reserved', 'Section: IN; Blz. 12', 'Length: 1873 words', 'Byline: Casper Thomas', 'Body']
DEBUG: First few lines of the text:
['Page 1 of 4', 'Groen, geel, blauw en slordig Kunst Nieuw Parijs', 'Groen, geel, blauw en slordig; Kunst Nieuw Parijs', 'De Groene Amsterdammer', '27 februari 2025', '© Copyright 2025 De Groene Amsterdammer All Rights Reserved', 'Section: IN; Blz. 56', 'Length: 1865 words', 'Byline: Koen Kleijn', 'Body']
DEBUG: First few lines of the text:
['Page 1 of 5', 'Wel je diploma, niet naar Rome? Corona De verloren Grand Tour', 'Wel je diploma, niet naar Rome?; Corona De verloren Grand Tour', 'De Groene Amsterdammer', '4 juni 2020', '© Copyright 2020 De Groene Amsterdammer All Rights Reserved', 'S

,title,outlet,date,authors,body
0,Irrationaliteit is het punt De handelsoorlog v...,GA,2025-04-17 00:00:00,Casper Thomas,Trumps gejojo met handelstarieven heeft niets ...
1,"Groen, geel, blauw en slordig Kunst Nieuw Pari...",GA,2025-02-27 00:00:00,Koen Kleijn,De impressionisten die zich in de negentiende ...
2,"Wel je diploma, niet naar Rome? Corona De verl...",GA,2020-06-04 00:00:00,Anne Branbergen,"Rome zonder toeristen, vanwege corona, is een ..."
3,Het geboortehuis van Herman Gorter essay Het g...,GA,2018-10-18 00:00:00,Jacob Groot,1. Dit is de genealogie van een virtualiteit. ...
4,Verstrengelde dimensies Holland Festival: Muzi...,GA,2015-04-12 00:00:00,Joep Christenhusz,Op het podium hangen half doorschijnende scher...
...,...,...,...,...,...
853,Niets nieuws Technologica Marcel Möring Niets ...,GA,2023-04-20 00:00:00,beeld Han Hoogerbrugge,Wim de Bie deelde zijn vertedering voor overbo...
854,Susan Wojcicki 5 juli 1968 - 9 augustus 2024 H...,GA,1968-07-05 00:00:00,Eva Hofman,"Susan Wojcicki, CEO van Google, 'machtigste vr..."
855,Op digitaal consult Zorg Chatten met de dokter...,GA,2020-04-23 00:00:00,Pierre de Winter,Vanuit zijn Amsterdamse praktijk heeft Vladan ...
856,Kwebbelkop Kwebbelkop,GA,2017-10-19 00:00:00,WALTER VAN DER KOOI,TELEVISIE Goudzoekers in YouTubeland\n \nIn...


In [ ]:
# Calculate word count for each row in the 'body' column
df_cleaned['word_count'] =df_cleaned['body'].apply(lambda x: len(str(x).split()))

# Display the DataFrame with the new 'word_count' column
df_cleaned.head()

/var/folders/w9/nj145rf562g6qb__l0s20l440000gn/T/ipykernel_36454/614310843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['word_count'] =df_cleaned['body'].apply(lambda x: len(str(x).split()))


,title,outlet,date,authors,body,word_count
0,Irrationaliteit is het punt De handelsoorlog v...,GA,2025-04-17 00:00:00,Casper Thomas,Trumps gejojo met handelstarieven heeft niets ...,1769
1,"Groen, geel, blauw en slordig Kunst Nieuw Pari...",GA,2025-02-27 00:00:00,Koen Kleijn,De impressionisten die zich in de negentiende ...,1792
2,"Wel je diploma, niet naar Rome? Corona De verl...",GA,2020-06-04 00:00:00,Anne Branbergen,"Rome zonder toeristen, vanwege corona, is een ...",3306
3,Het geboortehuis van Herman Gorter essay Het g...,GA,2018-10-18 00:00:00,Jacob Groot,1. Dit is de genealogie van een virtualiteit. ...,3075
4,Verstrengelde dimensies Holland Festival: Muzi...,GA,2015-04-12 00:00:00,Joep Christenhusz,Op het podium hangen half doorschijnende scher...,1672


In [8]:
df_cleaned.shape

(7310, 6)

In [ ]:
# Ensure 'date' is treated as string before regex
df['date'] = df['date'].apply(
    lambda x: str(x) + " 00:00:00" if pd.notna(x) and not re.search(r'\d{2}:\d{2}:\d{2}', str(x)) else str(x)
)

# Convert to datetime
df['date'] = pd.to_datetime(df['date'].str.strip(), errors='coerce')

In [10]:
df['year'] = df['date'].dt.year

In [11]:
#check if there are any NaNs in the 'date' column
nan_exists = df['date'].isna().any()
print(f"Are there any NaNs in 'date'? {nan_exists}")

#count exactly how many NaNs there are
nan_count = df['date'].isna().sum()
print(f"Number of NaNs in 'date': {nan_count}")

Are there any NaNs in 'date'? False
Number of NaNs in 'date': 0


In [ ]:
#filter for 2000-2024
#build the mask of rows we WANT to keep
#mask = df['date'].dt.year.between(2000, 2024)

#drop everything else, in place
#df.drop(df.index[~mask], inplace=True)


In [12]:
counts_per_year = df['year'].value_counts().sort_index()
print(counts_per_year)

1944       1
2015     120
2016     132
2017     126
2018     170
2019     496
2020     518
2021     624
2022     630
2023    1276
2024    1832
2025    1384
2050       1
Name: year, dtype: int64


In [ ]:
output_folder = os.path.dirname(folder_path)
#save to folder
df_cleaned.to_csv(os.path.join(output_folder, output_csv))
